In [6]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import os


nx, ny = 120, 120
dx = dy = 1.0
dt = 0.05

x = np.linspace(0, 1, nx)
y = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x, y)


D = 0.15 + 0.1 * np.exp(-((X - 0.5)**2 + (Y - 0.5)**2) * 10)


decay = 0.01


sigma = 0.05  # width of the blob
C = 100 * np.exp(-((X - 0.5)**2 + (Y - 0.5)**2) / (2 * sigma**2))


os.makedirs("outputs", exist_ok=True)


fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(C.T, cmap='hot', origin='lower', vmin=0, vmax=100, animated=True)
plt.colorbar(im, label='Concentration')
title = ax.set_title("Drug Diffusion (t = 0.00)")

def laplacian(Z):
    """Compute Laplacian using finite differences (periodic boundaries)."""
    return (
        np.roll(Z, 1, axis=0) + np.roll(Z, -1, axis=0) +
        np.roll(Z, 1, axis=1) + np.roll(Z, -1, axis=1) -
        4 * Z
    ) / (dx * dx)

def update(frame):
    global C
    diffusion = D * laplacian(C)
    reaction = -decay * C
    C = C + dt * (diffusion + reaction)

  
    C[0, :] = 0
    C[-1, :] = 0
    C[:, 0] = 0
    C[:, -1] = 0

    im.set_array(C.T) 
    title.set_text(f"Drug Diffusion (t = {frame*dt:.2f})")
    return [im, title]


ani = FuncAnimation(fig, update, frames=200, interval=50, blit=False)


print("Saving animation...")
ani.save("outputs/diffusion.gif", writer=PillowWriter(fps=20))
print("Saved to outputs/diffusion.gif")


plt.pause(0.001)
plt.show(block=True)

Saving animation...
Saved to outputs/diffusion.gif
